<a href="https://colab.research.google.com/github/langurmonkey/notebooks/blob/main/GaiaSky_Qwen3_5_4B_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaia Sky fine tune (Qwen 3.5 4B)
To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!

To install Unsloth on your local device, follow [Unsloth's guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

### Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B", # Use the text-only version
    max_seq_length = 4096,                      # Gaia Sky files are long; set accordingly
    load_in_4bit = True,                        # Keep 4bit for speed/memory
    dtype = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

For this Gaia Sky fine tuning, we only use language.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    finetune_vision_layers = False,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


<a name="Data"></a>
### Data Prep
We'll be using the Gaia Sky Q&A dataset. The goal is to turn the model into a Gaia Sky expert.

The full dataset is [here](https://huggingface.co/datasets/Langurmonkey/gaiasky-training-dataset).

In [ ]:
from datasets import load_dataset
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # Format as a standard ChatML conversation
        text = f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("Langurmonkey/gaiasky-training-dataset", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/11295 [00:00<?, ? examples/s]

Map:   0%|          | 0/11295 [00:00<?, ? examples/s]

Let's take an overview look at the dataset. We shall see what the 3rd image is, and what caption it had.

In [ ]:
dataset

Dataset({
    features: ['instruction', 'output', 'source_file', 'text'],
    num_rows: 11295
})

In [ ]:
dataset[30]["instruction"]

'How should documentation updates be distinguished from code changes in a merge request workflow?'

In [ ]:
dataset[30]["output"]

'Documentation updates require separate handling compared to core source code:\n\n1. If changes are needed specifically for the Gaia Sky project documentation, a **new merge request** must be created in the dedicated [documentation project](https://codeberg.org/gaiasky/gaiasky-docs).\n2. Documentation files like `README` or `ACKNOWLEDGEMENTS` fall under the `docs` commit type.\n3. This separation ensures that documentation changes are reviewed and managed independently from functional code updates.'

Let's now tokenize the dataset.

In [ ]:
# Define the formatting and tokenization function
def tokenize_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]

    # Qwen-VL models expect a specific conversation format or ChatML
    # We must explicitly tell the tokenizer that 'images' is None
    texts = [
        f"<|im_start|>user\n{i}<|im_end|>\n<|im_start|>assistant\n{o}<|im_end|>"
        for i, o in zip(instructions, outputs)
    ]

    # Use the processor/tokenizer with images=None
    tokenized = tokenizer(
        text = texts,
        images = None,         # No images, only text
        truncation = True,
        max_length = 4096,
        padding = False,       # Packing will handle padding
    )
    return tokenized

# Map the dataset and REMOVE all raw columns
# This leaves only input_ids and attention_mask, which fixes your error
dataset = dataset.map(
    tokenize_func,
    batched = True,
    remove_columns = dataset.column_names
)

Map:   0%|          | 0/11295 [00:00<?, ? examples/s]

<a name="Train"></a>
### Train the model
Now let's train our model. We use the standard `SFTTrainer`. We also enable packing, which is an Unsloth feature that groups multiple short Q&A pairs into a single 4096-token block, making the 1,700-file training run significantly faster.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    max_seq_length = 4096,
    packing = True, # Packing will now work perfectly with input_ids
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 120,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)


Unsloth: Switching to float32 training since model cannot work with float16


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
7.225 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,295 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Step,Training Loss
1,2.244549
2,2.257103
3,2.495746
4,1.928137
5,2.159629
6,2.263581
7,2.197516
8,1.490903
9,2.113878
10,1.867587


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1260.7976 seconds used for training.
21.01 minutes used for training.
Peak reserved memory = 7.225 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 49.612 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input.


In [ ]:
# 1. Enable for inference
FastLanguageModel.for_inference(model)

instruction = "Explain how to implement a custom celestial body in Gaia Sky using the Java API."

# 2. Format using ChatML
messages = [
    {"role": "user", "content": instruction}
]

# 3. Apply template
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True, tokenize = False)

# 4. Tokenize specifically with images=None
inputs = tokenizer(
    text = [input_text],
    images = None,
    return_tensors = "pt",
).to("cuda")

# 5. Generate
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)

_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.7,
    min_p = 0.1
)

To create a custom celestial body, you must extend the `CelestialBody` class and implement the required methods. The core logic involves defining the body's position, size, and visual properties.

```java
import com.gaiasky.core.model.CelestialBody;
import com.gaiasky.core.model.CelestialBodyData;
import com.gaiasky.core.model.CelestialBodyDataData;
import com.gaiasky.core.model.CelestialBodyDataDataData;
import com.gaiasky.core.model.CelestialBodyDataDataDataData;
import com.gaiasky.core.model.CelestialBodyDataDataDataDataData;
import com.gaiasky.core.model.CelestialBodyDataDataDataDataDataData;
import com.gaiasky.core.model.CelestialBodyDataDataDataDataDataDataData;
import com.gaiasky.core.model.CelestialBodyDataDataDataDataDataDataDataData;
import com.gaiasky.core.model.CelestialBodyDataDataDataDataDataDataDataDataData;
import com.gaiasky.core.model.CelestialBodyDataDataDataDataDataDataDataDataDataData;
import com.gaiasky.core.model.CelestialBodyDataDataDataDataDataDataDataDataDataD

KeyboardInterrupt: 

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")

['qwen_lora/processor_config.json']

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to q4_k_m GGUF
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
if True: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if True: model.push_to_hub_gguf("Langurmonkey/Qwen_gaiasky_finetune", tokenizer, quantization_method = "q4_k_m", token = hf_token)


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 2 files from cache to `qwen_finetune`:   0%|          | 0/2 [00:00<?, ?it/s]
Unsloth: Copying 2 files from cache to `qwen_finetune`:  50%|█████     | 1/2 [04:28<04:28, 268.01s/it]
Unsloth: Copying 2 files from cache to `qwen_finetune`: 100%|██████████| 2/2 [07:44<00:00, 232.24s/it]


Successfully copied all 2 files from cache to `qwen_finetune`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 16710.37it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [14:35<00:00, 437.99s/it]


Unsloth: Merge process complete. Saved to `/content/qwen_finetune`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen_finetune_gguf/Qwen3.5-4B.F16.gguf', 'qwen_finetune_gguf/Qwen3.5-4B.F16-mmproj.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


And we're done! If you have any questions on Unsloth, they have a [Discord](https://discord.gg/unsloth) channel!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.
